# Edge IIoT - Binary Classification


In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
import torch

print(torch.__version__) # X.XX.X+cuXXX / If X.XX.X+cpu it won't work
print(torch.cuda.is_available()) # False
print(torch.version.cuda) # None or mismatched version
print(torch.cuda.device_count()) # 0

2.5.1+cu121
True
12.1
1


In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import DATASETS, SEED
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['processed_path']}\n")

filename = "ML-EdgeIIoT-dataset_clean"

csv_path = config['processed_path'] / f"{filename}.pkl"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_pickle(csv_path)

print(f"\nDataset loaded with shape: {df.shape}")


--- Dataset Information ---
Name: EDGE_IIOT
Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed

CSV Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed/ML-EdgeIIoT-dataset_clean.pkl

Loading dataset... (This may take a while)

Dataset loaded with shape: (152591, 62)


## 1. Undersampling and Balancing

In [4]:
target     = 'Attack_label'
target_str = 'Attack_type'

In [5]:
y_str = df[target_str]
df_no_label = df.drop(columns=[target_str])

In [6]:
MAX_PRESENCE = 0.02

counts = y_str.value_counts()
total_rows = len(y_str)

print(counts)
print("\nTOTAL: ", total_rows)

Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                13307
Ransomware               10923
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
Backdoor                 10195
Vulnerability_scanner    10075
Port_Scanning            10071
XSS                      10051
Password                  9989
DDoS_TCP                  6011
MITM                      1028
Fingerprinting            1001
Name: count, dtype: int64

TOTAL:  152591


In [7]:
print(f"{'Attack Type':<25} | {'Old Count':<15} | {'New Count':<15}\n" + "-"*55)

sampling_strategy = {}
for attack_type, count in counts.items():
    current_presence = count / total_rows

    if current_presence > MAX_PRESENCE:
        new_count = int(total_rows * MAX_PRESENCE)
    else:
        new_count = count

    sampling_strategy[attack_type] = new_count
    print(f"{attack_type:<25} | {count:<15} | {new_count:<15}")


print("-"*55 + f"\n{'TOTAL':<25} | {counts.sum():<15} | {sum(sampling_strategy.values()):<15}")


Attack Type               | Old Count       | New Count      
-------------------------------------------------------
Normal                    | 24301           | 3051           
DDoS_UDP                  | 14498           | 3051           
DDoS_ICMP                 | 13307           | 3051           
Ransomware                | 10923           | 3051           
DDoS_HTTP                 | 10561           | 3051           
SQL_injection             | 10311           | 3051           
Uploading                 | 10269           | 3051           
Backdoor                  | 10195           | 3051           
Vulnerability_scanner     | 10075           | 3051           
Port_Scanning             | 10071           | 3051           
XSS                       | 10051           | 3051           
Password                  | 9989            | 3051           
DDoS_TCP                  | 6011            | 3051           
MITM                      | 1028            | 1028           
Fingerprinting

In [8]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=SEED)
df_no_label, y_str = rus.fit_resample(df_no_label, y_str)

print(df_no_label.shape)

(41692, 61)


In [9]:
# X/y split
X     = df_no_label.drop(columns=[target])
y     = df_no_label[target]

## 1. Pre-processing

In [10]:
# Select only numeric
print("--- Non-numeric cols to drop ---\n\n", X.select_dtypes(include=['str', 'object', 'category']).columns)

X = X.select_dtypes(include=['number'])

print("\n\nRemaining categorical cols:", len(X.select_dtypes(include=['str', 'object', 'category']).columns))

--- Non-numeric cols to drop ---

 Index(['http.file_data', 'http.referer', 'http.request.path',
       'http.request.uri.query', 'http.request.version', 'mqtt.msg', 'proto'],
      dtype='str')


Remaining categorical cols: 0


In [11]:
# Remove env-specific columns
cols_to_drop = [
    'frame.time.delta', 'frame.time.order',
    *[c for c in X.columns if c.startswith('ip.src_category') or c.startswith('ip.dst_category')]
]
X = X.drop(columns=cols_to_drop, errors='ignore')

In [12]:
print(f"NaN values in target variable: {y.isna().sum()}")
print(f"NaN values in features: {X.isna().sum().sum()}")

NaN values in target variable: 0
NaN values in features: 0


In [13]:
X_train, X_test, y_train, y_test, y_str_train, y_str_test = train_test_split(
    X, y, y_str, test_size=0.2, random_state=SEED, stratify=y_str
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (33353, 43)
X_test shape: (8339, 43)


In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_results = {}

## 2. LazyPredict
[Docs](https://pypi.org/project/lazypredict/)

In [15]:
from lazypredict.Supervised import LazyClassifier

# With categorical encoding, timeout, cross-validation, and GPU
clf = LazyClassifier(
    verbose=1,                          # Show progress
    ignore_warnings=True,               # Suppress warnings
    custom_metric=None,                 # Use default metrics
    predictions=False,                  # Don't Return predictions
    classifiers='all',                  # Use all available classifiers
    timeout=60,                         # Max time per model in seconds
    cv=5,                               # Cross-validation folds (optional)
)

models, _ = clf.fit(X_train, X_test, y_train, y_test)
print("\n--- Models Evaluated ---")

  0%|          | 0/32 [00:00<?, ?it/s]

/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1


--- Models Evaluated ---


In [16]:
display(models)

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
LGBMClassifier,0.988368,0.921247,0.991908,0.987912,0.988486,0.988368,0.988847,0.001383,0.923804,0.009428,0.992813,0.000927,0.988412,0.001499,0.988981,0.001349,0.988847,0.001383,298.006803
XGBClassifier,0.988008,0.921053,0.991889,0.987555,0.988058,0.988008,0.988547,0.001360,0.922888,0.009116,0.992622,0.000943,0.988104,0.001472,0.988649,0.001337,0.988547,0.001360,2.108082
ExtraTreeClassifier,0.987768,0.919413,0.989251,0.987295,0.987822,0.987768,0.987407,0.002250,0.915479,0.016553,0.991254,0.001577,0.986855,0.002494,0.987536,0.002152,0.987407,0.002250,1.319126
KNeighborsClassifier,0.987888,0.917213,0.920460,0.987381,0.988044,0.987888,0.988247,0.001437,0.920274,0.009929,0.933348,0.027454,0.987770,0.001565,0.988377,0.001397,0.988247,0.001437,1.338996
DecisionTreeClassifier,0.984291,0.892623,0.983724,0.983412,0.984553,0.984291,0.985219,0.001367,0.899208,0.009413,0.985181,0.000999,0.984442,0.001519,0.985444,0.001320,0.985219,0.001367,1.478646
BaggingClassifier,0.984291,0.892623,0.985737,0.983412,0.984553,0.984291,0.985219,0.001323,0.899396,0.009113,0.986750,0.001304,0.984446,0.001470,0.985435,0.001278,0.985219,0.001323,1.650526
RandomForestClassifier,0.984171,0.891803,0.985350,0.983278,0.984437,0.984171,0.985189,0.001402,0.898814,0.009548,0.987632,0.001801,0.984405,0.001557,0.985424,0.001358,0.985189,0.001402,2.533682
ExtraTreesClassifier,0.984171,0.891803,0.992340,0.983278,0.984437,0.984171,0.985938,0.001668,0.904116,0.011660,0.992466,0.000940,0.985234,0.001852,0.986144,0.001610,0.985938,0.001668,1.228391
AdaBoostClassifier,0.980213,0.866264,0.962406,0.978811,0.980526,0.980213,0.980931,0.000823,0.872935,0.005490,0.965658,0.004570,0.979679,0.000936,0.981121,0.000822,0.980931,0.000823,2.279737


In [17]:
display(models.sort_values(by='F1 Score CV Mean', ascending=False).head(5))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
LGBMClassifier,0.988368,0.921247,0.991908,0.987912,0.988486,0.988368,0.988847,0.001383,0.923804,0.009428,0.992813,0.000927,0.988412,0.001499,0.988981,0.001349,0.988847,0.001383,298.006803
XGBClassifier,0.988008,0.921053,0.991889,0.987555,0.988058,0.988008,0.988547,0.001360,0.922888,0.009116,0.992622,0.000943,0.988104,0.001472,0.988649,0.001337,0.988547,0.001360,2.108082
KNeighborsClassifier,0.987888,0.917213,0.920460,0.987381,0.988044,0.987888,0.988247,0.001437,0.920274,0.009929,0.933348,0.027454,0.987770,0.001565,0.988377,0.001397,0.988247,0.001437,1.338996
ExtraTreeClassifier,0.987768,0.919413,0.989251,0.987295,0.987822,0.987768,0.987407,0.002250,0.915479,0.016553,0.991254,0.001577,0.986855,0.002494,0.987536,0.002152,0.987407,0.002250,1.319126
ExtraTreesClassifier,0.984171,0.891803,0.992340,0.983278,0.984437,0.984171,0.985938,0.001668,0.904116,0.011660,0.992466,0.000940,0.985234,0.001852,0.986144,0.001610,0.985938,0.001668,1.228391
